# Ensemble eval — mini-ensemble seed13 + seed42 (эксперимент 3)

Оценка мини-ансамбля на испорченном dev: пул beam-кандидатов от двух чекпоинтов
(seed13 + seed42) + reference-free **MBR-chrF** селектор. Сравниваем с одиночной
лучшей моделью (seed13). Также считаем COMET на ансамбле (метрика для отчёта).

Настройка: **GPU T4 x2**, **Internet On**, Secret `HF_TOKEN`.

In [ ]:
BRANCH = "ml-dev"
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!rm -rf /kaggle/working/repo
!git clone --branch {BRANCH} https://github.com/ObjoradDdd/ml-hits-3-lab.git /kaggle/working/repo
%cd /kaggle/working/repo/ml
!pip install -q -e . sacrebleu

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login, whoami
login(os.environ["HF_TOKEN"])
USER = whoami()["name"]
S13 = f"{USER}/akkadian-byt5-full-seed13"
S42 = f"{USER}/akkadian-byt5-full-seed42"
print(S13, S42)

In [ ]:
# нужен dev.csv -> строим корпус из данных соревнования
COMP = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
!mkdir -p data && cp {COMP}/train.csv {COMP}/test.csv \
    {COMP}/published_texts.csv {COMP}/Sentences_Oare_FirstWord_LinNum.csv data/
!python -m akkadian_nmt.data_prep --data_dir=./data --out_dir=./data/processed

In [ ]:
# одиночная лучшая (seed13, beam=8) — контрольная точка
!python -m akkadian_nmt.evaluate run --model_dirs={S13} --num_beams=8 --max_samples=200

In [ ]:
# АНСАМБЛЬ: 4 beam-кандидата от каждой модели -> 8 в пуле -> MBR-chrF выбор
import json
MODELS = json.dumps([S13, S42])  # '["kirmala/...seed13","kirmala/...seed42"]'
!python -m akkadian_nmt.evaluate run --model_dirs='{MODELS}' \
    --num_beams=4 --candidates_per_model=4 --max_samples=200 \
    --out_file=/kaggle/working/ensemble_dev.json

In [ ]:
# COMET на предсказаниях ансамбля (для отчёта). Запускаем последним.
!pip install -q unbabel-comet
!python -m akkadian_nmt.evaluate comet_score_file --pred_file=/kaggle/working/ensemble_dev.json